#### Preparing dataset for the model

In [ ]:
import numpy as np
from torch.utils.data import Dataset,DataLoader
import torch
import tiktoken

In [ ]:
text="""The Internet is a global system that connects computers and devices worldwide, enabling communication, information sharing, 
and access to digital services. Connects people and devices globallyEnables communication, learning, and business
Supports digital services and cloud platforms  Operates using standard communication protocols
Forms the backbone of modern digital life alone """

tokens=text.split(" ")

In [ ]:
for i in range(5):
    print(tokens[i*4:(i+1)*4])

In [ ]:
def sliding_window(context_size:int,text:str):
    tokens=text.split(" ")
    nrows=int(np.ceil(len(tokens)/context_size))
    window=[]
    for i in range(nrows):
        window.append(tokens[i*context_size:(i+1)*context_size])
    return window    

In [ ]:
sliding_window(5,text)

In [ ]:
class GPT2Tokenizer(Dataset):
    def __init__(self,text,tokenizer,context_size,strides):
        self.input_ids=[]
        self.target_ids=[]
        tokens=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
        for i in range(0,len(tokens)-context_size,strides):
            in_tokens=tokens[i:i+context_size] ## input token chunck
            target_tokens=tokens[i+1:i+context_size+1] ## target token chunk
            self.input_ids.append(torch.tensor(in_tokens))
            self.target_ids.append(torch.tensor(target_tokens))
    def __len__(self):
        return len(self.input_ids)  
    def __getitem__(self, index):
        return self.input_ids[index],self.target_ids[index] 

             

In [ ]:
def create_dataloader(txt,batch_size=4,max_length=256,strides=128,Suffle=True,drop_last=True,num_workers=0):
    tokeizer=tiktoken.get_encoding("gpt2")
    datset=GPT2Tokenizer(txt,tokenizer=tokeizer,context_size=max_length,strides=strides)
    dataloader=DataLoader(dataset=datset,batch_size=batch_size,shuffle=Suffle,drop_last=drop_last,num_workers=num_workers)
    return dataloader

In [ ]:
dataloder=create_dataloader(text,batch_size=4,max_length=4,strides=2,Suffle=True)

In [ ]:
data=iter(dataloder)
first_batch=next(data)
first_batch